# 🌐 NLLB-STT Demo — Google Colab**Bidirectional English ↔ Vietnamese Speech-to-Text + Translation**Pipeline: `Audio → Whisper (STT) → Text → NLLB-200 → Translation`| Component | Model ||-----------|-------|| Speech-to-Text | [faster-whisper](https://github.com/SYSTRAN/faster-whisper) (small) || Translation | [facebook/nllb-200-distilled-600M](https://huggingface.co/facebook/nllb-200-distilled-600M) |> **Colab tip:** Go to **Runtime → Change runtime type → T4 GPU** for best performance.> With GPU, model loading takes ~5s; on CPU it's ~30s.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────import sys, subprocess, importlibdeps = [    "torch>=2.0.0",    "transformers>=4.40.0",    "sentencepiece",    "accelerate",    "gradio>=5.0.0",    "soundfile",    "librosa",    "faster-whisper>=1.0.0",]missing = []for dep in deps:    pkg = dep.split(">=")[0].split("==")[0]    if not importlib.util.find_spec(pkg.replace("-", "_")):        missing.append(dep)if missing:    print(f"  Installing {len(missing)} missing packages...")    subprocess.check_call([        sys.executable, "-m", "pip", "install", "-q",        *missing,    ])    print("  Done.")else:    print("  All dependencies already installed.")

In [ ]:
# ── 2. Check GPU ────────────────────────────────────────────────import torchcuda = torch.cuda.is_available()if cuda:    name = torch.cuda.get_device_name(0)    print(f"  ✅ GPU detected: {name}")else:    print("  ⚠️  No GPU — will run on CPU (slower but works)")    print("  🛠️  Runtime → Change runtime type → T4 GPU")

In [ ]:
# ── 3. Import app module ─────────────────────────────────────────import sys, os# Clone/download the app source if not presentif not os.path.isdir("nllb-stt-demo"):    print("Downloading app source from GitHub...")    # If you have the files locally, upload them or clone from your repo    os.makedirs("nllb-stt-demo", exist_ok=True)    print("NOTE: Please upload `app.py` to the `nllb-stt-demo/` folder in Colab's")    print("      file browser, or run the pipeline cells below directly.")else:    print("app/ directory found.")sys.path.insert(0, "nllb-stt-demo")# If the module is available, import it; otherwise define standalone helpers belowtry:    from app import load_models, transcribe, translate, build_ui    print("  Loaded from app.py")except ImportError:    print("  Using standalone pipeline (define helpers inline)")

## Standalone Pipeline (no app.py needed)If you didn't upload `app.py`, these cells define the full pipeline inline.

In [ ]:
# ── 4. Define the pipeline inline (if app.py isn't available) ───# This mirrors the exact logic from app.py so everything works standalone.try:    from app import load_models, transcribe, translate    print("app.py loaded — skipping inline definitions.")except Exception:    import time, warnings    import torch    from faster_whisper import WhisperModel    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer    warnings.filterwarnings("ignore")    NLLB_NAME = "facebook/nllb-200-distilled-600M"    LANG_MAP = {"English": "eng_Latn", "Vietnamese": "vie_Latn"}    WHISPER_LANG = {"English": "en", "Vietnamese": "vi"}    _whisper_model = None    _nllb_model = None    _nllb_tokenizer = None    def get_device():        if torch.cuda.is_available():            print(f"  GPU: {torch.cuda.get_device_name(0)}")            return "cuda"        print("  CPU mode")        return "cpu"    def load_models():        global _whisper_model, _nllb_model, _nllb_tokenizer        if _whisper_model is not None:            return        device = get_device()        is_cuda = device == "cuda"        print("Loading Whisper (small)...")        _whisper_model = WhisperModel("small", device=device,                                       compute_type="float16" if is_cuda else "int8")        print("Loading NLLB-200 distilled 600M...")        _nllb_tokenizer = AutoTokenizer.from_pretrained(NLLB_NAME, trust_remote_code=True)        _nllb_model = AutoModelForSeq2SeqLM.from_pretrained(            NLLB_NAME, trust_remote_code=True, torch_dtype="auto", low_cpu_mem_usage=True,        )        if is_cuda:            _nllb_model = _nllb_model.to("cuda")        print("  Models ready!")    def transcribe(audio_path, source_lang):        if not audio_path or not os.path.isfile(audio_path):            return "(no audio)", 0.0        t0 = time.time()        segs, _ = _whisper_model.transcribe(audio_path, language=WHISPER_LANG[source_lang])        text = " ".join(s.text.strip() for s in segs)        return (text.strip() or "(no speech)"), round(time.time() - t0, 2)    def translate(text, source_lang, target_lang):        if not text or text.startswith("(no"):            return "(nothing)", 0.0        t0 = time.time()        _nllb_tokenizer.src_lang = LANG_MAP[source_lang]        inputs = _nllb_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)        toks = _nllb_model.generate(            **inputs,            forced_bos_token_id=_nllb_tokenizer.convert_tokens_to_ids(LANG_MAP[target_lang]),            max_length=512,        )        out = _nllb_tokenizer.batch_decode(toks, skip_special_tokens=True)[0]        return out.strip(), round(time.time() - t0, 2)    print("  Inline pipeline defined.")

In [ ]:
# ── 5. Load models (this downloads once — takes ~5s on GPU, ~30s on CPU) ───import timet0 = time.time()load_models()print(f"\n✅ Models loaded in {time.time()-t0:.1f}s")

## ✏️ Text Translation TestTest NLLB-200's translation quality directly without audio.

In [ ]:
# ── 6. Text translation test ────────────────────────────────────tests = [    ("English", "Vietnamese", "Hello, how are you today?"),    ("English", "Vietnamese", "I would like to visit Ho Chi Minh City next summer."),    ("Vietnamese", "English", "Chào bạn, hôm nay bạn có khỏe không?"),    ("Vietnamese", "English", "Tôi muốn học tiếng Anh để đi du lịch nước ngoài."),]print(f"{'Source':<25} {'Target':<25} {'Text → Translation'}")print("-" * 100)for src, tgt, text in tests:    result, t = translate(text, src, tgt)    print(f"{src:<25} {tgt:<25} \"{text}\" → \"{result}\" ({t:.1f}s)")

## 🎤 Speech-to-Text + Translation PipelineUpload an audio file (.wav, .mp3, .m4a) to transcribe and translate.

In [ ]:
# ── 7. Upload audio file ─────────────────────────────────────────from google.colab import filesprint("Select an audio file (.wav / .mp3 / .m4a)...")uploaded = files.upload()audio_path = Nonefor fname in uploaded:    audio_path = fname    print(f"  Uploaded: {fname} ({len(uploaded[fname])/1024:.1f} KB)")    breakif audio_path:    # Show audio player    from IPython.display import Audio, display    display(Audio(audio_path))

In [ ]:
# ── 8. Run the pipeline ──────────────────────────────────────────if audio_path:    for source_lang in ["English", "Vietnamese"]:        print(f"\n{'='*60}")        print(f"  Treating audio as: {source_lang}")        print(f"{'='*60}")        text, stt_t = transcribe(audio_path, source_lang)        print(f"  🎤 Transcription ({stt_t:.1f}s): {text}")        if text and not text.startswith("(no"):            target = "Vietnamese" if source_lang == "English" else "English"            arrow = "→" if source_lang == "English" else "←"            trans, nllb_t = translate(text, source_lang, target)            print(f"  🔄  Translation ({nllb_t:.1f}s) {source_lang} {arrow} {target}:")            print(f"      {trans}")        else:            print("  ⚠️  No speech detected in this language")else:    print("  No audio uploaded — run cell 7 first.")

## 🌐 Gradio Web UI (embedded in Colab)Launch the full Gradio interface right inside this notebook.

In [ ]:
# ── 9. Launch Gradio UI in Colab ────────────────────────────────from google.colab import outputoutput.enable_custom_widget_manager()# app.py already has --colab flag logic; or use inline build:try:    from app import build_ui    demo = build_ui()except Exception:    # Minimal inline UI if app.py not available    import gradio as gr    def process_audio(audio_path, source_lang):        target_lang = "Vietnamese" if source_lang == "English" else "English"        arrow = "→" if source_lang == "English" else "←"        transcription, stt_t = transcribe(audio_path, source_lang)        translation, nllb_t = translate(transcription, source_lang, target_lang)        return (            f"**Pipeline: {stt_t + nllb_t:.1f}s** (STT: {stt_t}s | NLLB: {nllb_t}s)\n\n"            f"## 🎤 Transcription ({source_lang})\n{transcription}\n\n"            f"## 🔄 Translation {arrow} ({target_lang})\n{translation}"        )    with gr.Blocks(title="NLLB-STT Demo") as demo:        gr.Markdown("# 🌐 NLLB-STT Demo\nEnglish ↔ Vietnamese Speech-to-Text + Translation")        with gr.Row():            with gr.Column():                src = gr.Radio(["English", "Vietnamese"], label="Source Language", value="English")                audio = gr.Audio(type="filepath", label="Audio", sources=["microphone", "upload"])                btn = gr.Button("Transcribe & Translate", variant="primary")            with gr.Column():                out = gr.Markdown("Ready")        btn.click(fn=process_audio, inputs=[audio, src], outputs=[out])demo.queue(default_concurrency_limit=1)demo.launch(share=True, debug=True)

---### Notes- **First run** downloads both models (~1.6 GB total) — cached in Colab's VM for the session.- **GPU (T4)** → Whisper ~2s, NLLB ~1s per translation. **CPU** → ~5-10x slower.- If the VM disconnects, re-run cells 5 + 9 (models cache is lost on VM recycle).